# STEP 1 - Data understanding & preparation (CRISP-DM phases 2 and 3)
Bean there done that - Group 7

What this script does:
  1. Loads the 5 sheets from the raw Excel file
  2. Cleans them (consistent names, month numbers, extra columns we need)
  3. Saves tidy CSVs to data/clean/ so every later step starts from the same data
  4. Prints a data-quality report: things that look odd and need explaining

In [5]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "435bf210",
   "metadata": {},
   "source": [
    "# Step 1 — Business & data understanding, data preparation\n",
    "**Bean there done that · Group 7** · CRISP-DM phases 1–3\n",
    "\n",
    "This notebook:\n",
    "1. writes down what we know about the business (from the Week 3 assignment slides),\n",
    "2. maps every sheet of the dataset to a step in the company's operations,\n",
    "3. cleans the data and saves it to `data/clean/` (all later steps start from there),\n",
    "4. checks whether the data is consistent, and lists what we found and what we still need to ask.\n",
    "\n",
    "**Before running:** upload `Bean there done that data.xlsx` into this folder (ideally `data/raw/`)."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "ef9fde46",
   "metadata": {},
   "source": [
    "## 1. Business understanding\n",
    "\n",
    "**The question:** *The CEO wants to know if Bean there done that is ready to grow as of next year.*\n",
    "The company struggled to manage growth, so at the start of 2023 it stopped accepting new customers to create a steady state.\n",
    "\n",
    "**Our task (assignment guide):**\n",
    "- Use CRISP-DM, start with business and data understanding\n",
    "- Identify and visualise **KPIs for each part of the company operations**, and evaluate performance\n",
    "- Look for improvement opportunities **and** growth roadblocks: what breaks if they grow next year?\n",
    "- Advise the CEO on running operations better and on getting ready to grow\n",
    "\n",
    "**Company operations (slide 24):**\n",
    "\n",
    "`Sourcing → [Raw-material storage] → Roasting → Cooling → Packing → [Finished-product storage] → Delivery`\n",
    "\n",
    "| Step | What happens |\n",
    "|---|---|\n",
    "| Sourcing | Direct trade with farmers in Colombia, Uganda and Costa Rica. Green beans arrive in 70 kg bags, 12 bags per pallet, 12 pallets per 20 ft container. Packaging bought from ProPack (bags) and Boxster (boxes). |\n",
    "| Raw-material storage | Warehouse |\n",
    "| Roasting | 2 roasters: Probat P60 (215 kg/h, max batch 60 kg) and Probat Px120 (480 kg/h, max batch 120 kg) |\n",
    "| Cooling | After roasting, a separate step |\n",
    "| Packing | Fully automated packing line, 500 g or 1 kg bags |\n",
    "| Finished-product storage | Warehouse |\n",
    "| Delivery | E-com: PostNL, next-day except weekends & holidays, free shipping on orders > €55. B2B: own truck |\n",
    "\n",
    "**Company profile:** small NL roaster, ±1,000 t per year, <1% market share, domestic only, premium specialty coffee, mostly e-commerce with some B2B clients. 3 roasts × 2 bag sizes = 6 products."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "6c81e963",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-09T08:37:57.849712Z",
     "iopub.status.busy": "2026-10-09T08:37:57.849712Z",
     "iopub.status.idle": "2026-10-09T08:37:58.527692Z",
     "shell.execute_reply": "2026-10-09T08:37:58.527692Z"
    }
   },
   "outputs": [],
   "source": [
    "from pathlib import Path\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "\n",
    "pd.set_option(\"display.float_format\", \"{:,.2f}\".format)\n",
    "\n",
    "# ---- Business facts from the assignment slides (not in the Excel file) ----\n",
    "ROASTER_SPECS = pd.DataFrame({\n",
    "    \"Machine\": [\"Probat P60\", \"Probat Px120\"],\n",
    "    \"Rated kg/h\": [215, 480],\n",
    "    \"Max batch kg\": [60, 120],\n",
    "}).set_index(\"Machine\")\n",
    "KG_PER_BEAN_BAG = 70\n",
    "KG_PER_PALLET = 12 * KG_PER_BEAN_BAG          # 840 kg\n",
    "KG_PER_CONTAINER = 12 * KG_PER_PALLET         # 10,080 kg\n",
    "FREE_SHIPPING_ABOVE = 55                      # EUR, orders strictly above this ship free\n",
    "STATED_OUTPUT_T = 1000                        # \"±1.000 tonnes\" in company profile\n",
    "NL_HOLIDAYS_2023 = pd.to_datetime([           # no PostNL delivery\n",
    "    \"2023-01-01\", \"2023-04-10\", \"2023-04-27\", \"2023-05-18\",\n",
    "    \"2023-05-29\", \"2023-12-25\", \"2023-12-26\"])\n",
    "OPERATIONS = [\"Sourcing\", \"Raw-material storage\", \"Roasting\", \"Cooling\",\n",
    "              \"Packing\", \"Finished-product storage\", \"Delivery\"]\n",
    "MONTHS = [\"January\", \"February\", \"March\", \"April\", \"May\", \"June\", \"July\",\n",
    "          \"August\", \"September\", \"October\", \"November\", \"December\"]\n",
    "MONTH_NR = {m: i + 1 for i, m in enumerate(MONTHS)}"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "afc9952e",
   "metadata": {},
   "source": [
    "## 2. Load the data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "a403fe37",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-09T08:37:58.531205Z",
     "iopub.status.busy": "2026-10-09T08:37:58.531205Z",
     "iopub.status.idle": "2026-10-09T08:38:18.576998Z",
     "shell.execute_reply": "2026-10-09T08:38:18.575958Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Using C:\\Users\\floyd\\AppData\\Roaming\\Claude\\scratch-workspaces\\3cf67011-d217-456e-927e-9c7020195e99\\edb7be3c-9c78-471c-9f96-3e7c16929c9d\\scratch-2026-10-09-3f62ce\\bean-there-done-that\\data\\raw\\Bean there done that data.xlsx\n",
      "Inventory (raw material)               72 rows  10 columns\n",
      "Inventory (finished product)           36 rows  8 columns\n",
      "Roasting                               36 rows  15 columns\n",
      "E-com sales orders by product     240,352 rows  9 columns\n",
      "B2B orders                            273 rows  8 columns\n"
     ]
    }
   ],
   "source": [
    "ROOT = Path.cwd()\n",
    "RAW_NAME = \"Bean there done that data.xlsx\"\n",
    "found = [p for base in (ROOT, ROOT.parent) for p in base.rglob(RAW_NAME)]\n",
    "if not found:\n",
    "    raise FileNotFoundError(f\"Upload '{RAW_NAME}' into {ROOT} (e.g. data/raw/) and run again.\")\n",
    "RAW = found[0]\n",
    "CLEAN = ROOT / \"data\" / \"clean\"\n",
    "CLEAN.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "sheets = pd.read_excel(RAW, sheet_name=None)\n",
    "print(\"Using\", RAW)\n",
    "for name, df in sheets.items():\n",
    "    print(f\"{name:32} {len(df):>8,} rows  {df.shape[1]} columns\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "15a245d8",
   "metadata": {},
   "source": [
    "## 3. Data understanding: which sheet covers which operations step?\n",
    "\n",
    "The assignment asks for KPIs *per part of the operations*, so this mapping is the backbone of the analysis."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "33fab622",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-09T08:38:18.579765Z",
     "iopub.status.busy": "2026-10-09T08:38:18.579765Z",
     "iopub.status.idle": "2026-10-09T08:38:18.596243Z",
     "shell.execute_reply": "2026-10-09T08:38:18.595732Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Operations step</th>\n",
       "      <th>Sheet</th>\n",
       "      <th>What it tells us</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>Sourcing</td>\n",
       "      <td>Inventory (raw material)</td>\n",
       "      <td>Purchased per month, supplier, unit price</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>Raw-material storage</td>\n",
       "      <td>Inventory (raw material)</td>\n",
       "      <td>Stock level, scrapped + reason (DAM = damaged,...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>Roasting</td>\n",
       "      <td>Roasting</td>\n",
       "      <td>Batches, load/final weight, setup &amp; run time, ...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>Cooling</td>\n",
       "      <td>Roasting</td>\n",
       "      <td>Cooling time per batch</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>Packing</td>\n",
       "      <td>Roasting</td>\n",
       "      <td>Packing time per batch</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>Finished-product storage</td>\n",
       "      <td>Inventory (finished product)</td>\n",
       "      <td>Stock level, scrapped + reason (EXP = expired)</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>6</th>\n",
       "      <td>Delivery</td>\n",
       "      <td>E-com sales orders + B2B orders</td>\n",
       "      <td>Orders, value, order/delivery date, fees, ship...</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "            Operations step                            Sheet  \\\n",
       "0                  Sourcing         Inventory (raw material)   \n",
       "1      Raw-material storage         Inventory (raw material)   \n",
       "2                  Roasting                         Roasting   \n",
       "3                   Cooling                         Roasting   \n",
       "4                   Packing                         Roasting   \n",
       "5  Finished-product storage     Inventory (finished product)   \n",
       "6                  Delivery  E-com sales orders + B2B orders   \n",
       "\n",
       "                                    What it tells us  \n",
       "0          Purchased per month, supplier, unit price  \n",
       "1  Stock level, scrapped + reason (DAM = damaged,...  \n",
       "2  Batches, load/final weight, setup & run time, ...  \n",
       "3                             Cooling time per batch  \n",
       "4                             Packing time per batch  \n",
       "5     Stock level, scrapped + reason (EXP = expired)  \n",
       "6  Orders, value, order/delivery date, fees, ship...  "
      ]
     },
     "execution_count": 3,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "coverage = pd.DataFrame([\n",
    "    (\"Sourcing\",                 \"Inventory (raw material)\",     \"Purchased per month, supplier, unit price\"),\n",
    "    (\"Raw-material storage\",     \"Inventory (raw material)\",     \"Stock level, scrapped + reason (DAM = damaged, EXP = expired)\"),\n",
    "    (\"Roasting\",                 \"Roasting\",                     \"Batches, load/final weight, setup & run time, QC rejects\"),\n",
    "    (\"Cooling\",                  \"Roasting\",                     \"Cooling time per batch\"),\n",
    "    (\"Packing\",                  \"Roasting\",                     \"Packing time per batch\"),\n",
    "    (\"Finished-product storage\", \"Inventory (finished product)\", \"Stock level, scrapped + reason (EXP = expired)\"),\n",
    "    (\"Delivery\",                 \"E-com sales orders + B2B orders\", \"Orders, value, order/delivery date, fees, shipping cost, province/city\"),\n",
    "], columns=[\"Operations step\", \"Sheet\", \"What it tells us\"])\n",
    "coverage"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "68456230",
   "metadata": {},
   "source": [
    "## 4. Data preparation\n",
    "\n",
    "What we clean or add, per sheet:\n",
    "- consistent roast names (`Dark roast` / `Dark Roast` → `Dark`), month numbers\n",
    "- scrapped quantities as positive numbers, plus their € value\n",
    "- **Bag type 1 = 1 kg, Bag type 2 = 0.5 kg** (checked below via unit price)\n",
    "- derived columns we need for KPIs: batches, roaster hours, throughput, yield, reject rate, lead time in business days, free-shipping check"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "d787a1ad",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-09T08:38:18.599775Z",
     "iopub.status.busy": "2026-10-09T08:38:18.598762Z",
     "iopub.status.idle": "2026-10-09T08:38:19.104847Z",
     "shell.execute_reply": "2026-10-09T08:38:19.104847Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Cleaned: {'raw': 72, 'roasting': 36, 'finished': 36, 'ecom': 240352, 'b2b': 273}\n"
     ]
    }
   ],
   "source": [
    "def roast_name(s):\n",
    "    return s.str.strip().str.split().str[0].str.capitalize()\n",
    "\n",
    "# --- Raw material (Sourcing + Raw-material storage) ---\n",
    "raw = sheets[\"Inventory (raw material)\"].copy()\n",
    "raw[\"Month nr\"] = raw[\"Month\"].map(MONTH_NR)\n",
    "raw[\"Category\"] = np.where(raw[\"Product\"].str.contains(\"Beans\"), \"Green beans\", \"Packaging\")\n",
    "raw[\"Origin\"] = raw[\"Product\"].str.extract(r\"COO (\\w+(?: \\w+)?)\\)\")\n",
    "raw[\"Scrapped\"] = raw[\"Scrapped\"].abs()\n",
    "raw[\"Scrapped value\"] = raw[\"Scrapped\"] * raw[\"Unit price\"]\n",
    "raw[\"Reasoncode\"] = raw[\"Reasoncode\"].fillna(\"\")\n",
    "raw = raw.sort_values([\"Product\", \"Month nr\"]).reset_index(drop=True)\n",
    "beans = raw[\"Category\"] == \"Green beans\"\n",
    "raw.loc[beans, \"Containers purchased\"] = raw[\"Purchased\"] / KG_PER_CONTAINER\n",
    "raw.loc[beans, \"Pallets scrapped\"] = raw[\"Scrapped\"] / KG_PER_PALLET\n",
    "# Usage = opening stock + purchased - scrapped - closing stock (not available for January)\n",
    "raw[\"Used\"] = (raw.groupby(\"Product\")[\"Stock level\"].shift()\n",
    "               + raw[\"Purchased\"] - raw[\"Scrapped\"] - raw[\"Stock level\"])\n",
    "\n",
    "# --- Roasting (Roasting + Cooling + Packing) ---\n",
    "ro = sheets[\"Roasting\"].dropna(how=\"all\").copy()\n",
    "ro.columns = [\"Machine\", \"Month\", \"Roast\", \"Workdays\", \"Batches per day\",\n",
    "              \"Load kg\", \"Final kg\", \"Setup min\", \"Run min\", \"Cooling min\",\n",
    "              \"Packing min\", \"QC weight\", \"QC color\", \"QC profile\", \"Rejected batches\"]\n",
    "ro[\"Roast\"] = roast_name(ro[\"Roast\"])\n",
    "ro[\"Month nr\"] = ro[\"Month\"].map(MONTH_NR)\n",
    "ro[\"Batches\"] = ro[\"Workdays\"] * ro[\"Batches per day\"]\n",
    "ro[\"Cycle min\"] = ro[[\"Setup min\", \"Run min\", \"Cooling min\"]].sum(axis=1)\n",
    "ro[\"Roaster hours\"] = ro[\"Batches\"] * ro[\"Cycle min\"] / 60\n",
    "ro[\"Packing hours\"] = ro[\"Batches\"] * ro[\"Packing min\"] / 60\n",
    "ro[\"Kg loaded\"] = ro[\"Batches\"] * ro[\"Load kg\"]\n",
    "ro[\"Kg roasted\"] = ro[\"Batches\"] * ro[\"Final kg\"]\n",
    "ro[\"Kg good\"] = (ro[\"Batches\"] - ro[\"Rejected batches\"]) * ro[\"Final kg\"]\n",
    "ro[\"Yield\"] = ro[\"Final kg\"] / ro[\"Load kg\"]\n",
    "ro[\"Reject rate\"] = ro[\"Rejected batches\"] / ro[\"Batches\"]\n",
    "ro[\"Batch fill\"] = ro[\"Load kg\"] / ro[\"Machine\"].map(ROASTER_SPECS[\"Max batch kg\"])\n",
    "\n",
    "# --- Finished product (Finished-product storage) ---\n",
    "fin = sheets[\"Inventory (finished product)\"].copy()\n",
    "fin[\"Roast\"] = roast_name(fin[\"Product\"])\n",
    "fin[\"Month nr\"] = fin[\"Month\"].map(MONTH_NR)\n",
    "fin[\"Scrapped\"] = fin[\"Scrapped\"].abs()\n",
    "fin[\"Scrapped value\"] = fin[\"Scrapped\"] * fin[\"Unit price\"]\n",
    "fin[\"Reasoncode\"] = fin[\"Reasoncode\"].fillna(\"\")\n",
    "\n",
    "# --- E-com orders (Delivery) ---\n",
    "ec = sheets[\"E-com sales orders by product\"].dropna(subset=[\"Product\"]).copy()\n",
    "ec = ec.loc[:, ~ec.columns.astype(str).str.startswith(\"Unnamed\")]\n",
    "ec[\"Roast\"] = roast_name(ec[\"Product\"])\n",
    "ec[\"Bag kg\"] = ec[\"Bag type\"].map({1: 1.0, 2: 0.5})\n",
    "ec[\"Kg\"] = ec[\"Quantity\"] * ec[\"Bag kg\"]\n",
    "ec[\"Month nr\"] = ec[\"Order date\"].dt.month\n",
    "ec[\"Order weekday\"] = ec[\"Order date\"].dt.day_name()\n",
    "hol = NL_HOLIDAYS_2023.values.astype(\"datetime64[D]\")\n",
    "ec[\"Lead business days\"] = np.busday_count(\n",
    "    (ec[\"Order date\"] + pd.Timedelta(days=1)).values.astype(\"datetime64[D]\"),\n",
    "    (ec[\"Delivery date\"] + pd.Timedelta(days=1)).values.astype(\"datetime64[D]\"),\n",
    "    holidays=hol)\n",
    "ec[\"On time\"] = ec[\"Lead business days\"] <= 1\n",
    "ec[\"Should ship free\"] = ec[\"Value\"] > FREE_SHIPPING_ABOVE\n",
    "ec[\"Wrongly charged\"] = ec[\"Should ship free\"] & (ec[\"Delivery fee\"] > 0)\n",
    "ec[\"Net shipping cost\"] = ec[\"Shipping costs\"] - ec[\"Delivery fee\"]\n",
    "\n",
    "# --- B2B orders (Delivery) ---\n",
    "b2b = sheets[\"B2B orders\"].copy()\n",
    "b2b[\"Roast\"] = roast_name(b2b[\"Product\"])\n",
    "b2b[\"Bag kg\"] = b2b[\"Bag type\"].map({1: 1.0, 2: 0.5})\n",
    "b2b[\"Kg\"] = b2b[\"Quantity\"] * b2b[\"Bag kg\"]\n",
    "b2b[\"Month nr\"] = b2b[\"Delivery date\"].dt.month\n",
    "\n",
    "print(\"Cleaned:\", {n: len(d) for n, d in\n",
    "                   [(\"raw\", raw), (\"roasting\", ro), (\"finished\", fin), (\"ecom\", ec), (\"b2b\", b2b)]})"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7b75447a",
   "metadata": {},
   "source": [
    "### Check: bag type 1 = 1 kg, bag type 2 = 0.5 kg"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "id": "4e7500bd",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-09T08:38:19.106719Z",
     "iopub.status.busy": "2026-10-09T08:38:19.106719Z",
     "iopub.status.idle": "2026-10-09T08:38:19.151531Z",
     "shell.execute_reply": "2026-10-09T08:38:19.151026Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th>min</th>\n",
       "      <th>max</th>\n",
       "      <th>count</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Roast</th>\n",
       "      <th>Bag type</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th rowspan=\"2\" valign=\"top\">Dark</th>\n",
       "      <th>1</th>\n",
       "      <td>22.98</td>\n",
       "      <td>22.98</td>\n",
       "      <td>32591</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>11.49</td>\n",
       "      <td>11.49</td>\n",
       "      <td>75779</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th rowspan=\"2\" valign=\"top\">Light</th>\n",
       "      <th>1</th>\n",
       "      <td>26.50</td>\n",
       "      <td>26.50</td>\n",
       "      <td>14475</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>13.25</td>\n",
       "      <td>13.25</td>\n",
       "      <td>33584</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th rowspan=\"2\" valign=\"top\">Medium</th>\n",
       "      <th>1</th>\n",
       "      <td>25.90</td>\n",
       "      <td>25.90</td>\n",
       "      <td>25143</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>12.95</td>\n",
       "      <td>12.95</td>\n",
       "      <td>58780</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                  min   max  count\n",
       "Roast  Bag type                   \n",
       "Dark   1        22.98 22.98  32591\n",
       "       2        11.49 11.49  75779\n",
       "Light  1        26.50 26.50  14475\n",
       "       2        13.25 13.25  33584\n",
       "Medium 1        25.90 25.90  25143\n",
       "       2        12.95 12.95  58780"
      ]
     },
     "execution_count": 5,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "(ec.assign(**{\"Price per bag\": ec[\"Value\"] / ec[\"Quantity\"]})\n",
    "   .groupby([\"Roast\", \"Bag type\"])[\"Price per bag\"].agg([\"min\", \"max\", \"count\"]))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "05fce92d",
   "metadata": {},
   "source": [
    "## 5. Data quality: does the data add up?\n",
    "\n",
    "If the sheets are consistent we can trust cross-sheet KPIs (e.g. production vs sales).\n",
    "We test three stock flows:\n",
    "1. **Green beans:** beans used (from stock movements) = kg loaded into the roasters?\n",
    "2. **Finished product:** closing stock = opening stock + good roasted kg − kg sold − kg expired?\n",
    "3. **Packaging:** bags and boxes used = bags and boxes shipped?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "id": "3f97d50f",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-09T08:38:19.152994Z",
     "iopub.status.busy": "2026-10-09T08:38:19.152994Z",
     "iopub.status.idle": "2026-10-09T08:38:19.163333Z",
     "shell.execute_reply": "2026-10-09T08:38:19.163333Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Beans used (stock)</th>\n",
       "      <th>Kg loaded (roasting)</th>\n",
       "      <th>Difference</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Month nr</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>76,252.00</td>\n",
       "      <td>76,254.00</td>\n",
       "      <td>-2.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>87,082.00</td>\n",
       "      <td>87,084.00</td>\n",
       "      <td>-2.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>71,614.00</td>\n",
       "      <td>71,616.00</td>\n",
       "      <td>-2.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>79,708.00</td>\n",
       "      <td>79,708.00</td>\n",
       "      <td>-0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>6</th>\n",
       "      <td>83,687.00</td>\n",
       "      <td>83,689.00</td>\n",
       "      <td>-2.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>7</th>\n",
       "      <td>81,325.00</td>\n",
       "      <td>81,328.00</td>\n",
       "      <td>-3.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>8</th>\n",
       "      <td>87,630.00</td>\n",
       "      <td>87,632.00</td>\n",
       "      <td>-2.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>9</th>\n",
       "      <td>80,777.00</td>\n",
       "      <td>80,779.00</td>\n",
       "      <td>-2.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>10</th>\n",
       "      <td>79,867.00</td>\n",
       "      <td>79,869.00</td>\n",
       "      <td>-2.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>11</th>\n",
       "      <td>81,834.00</td>\n",
       "      <td>81,835.00</td>\n",
       "      <td>-1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>12</th>\n",
       "      <td>72,785.00</td>\n",
       "      <td>72,786.00</td>\n",
       "      <td>-1.00</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "          Beans used (stock)  Kg loaded (roasting)  Difference\n",
       "Month nr                                                      \n",
       "2                  76,252.00             76,254.00       -2.00\n",
       "3                  87,082.00             87,084.00       -2.00\n",
       "4                  71,614.00             71,616.00       -2.00\n",
       "5                  79,708.00             79,708.00       -0.00\n",
       "6                  83,687.00             83,689.00       -2.00\n",
       "7                  81,325.00             81,328.00       -3.00\n",
       "8                  87,630.00             87,632.00       -2.00\n",
       "9                  80,777.00             80,779.00       -2.00\n",
       "10                 79,867.00             79,869.00       -2.00\n",
       "11                 81,834.00             81,835.00       -1.00\n",
       "12                 72,785.00             72,786.00       -1.00"
      ]
     },
     "execution_count": 6,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "used_beans = raw[beans].groupby(\"Month nr\")[\"Used\"].sum()\n",
    "loaded = ro.groupby(\"Month nr\")[\"Kg loaded\"].sum()\n",
    "pd.DataFrame({\"Beans used (stock)\": used_beans, \"Kg loaded (roasting)\": loaded,\n",
    "              \"Difference\": used_beans - loaded}).loc[2:].round(0)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "id": "5256bf6a",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-09T08:38:19.165870Z",
     "iopub.status.busy": "2026-10-09T08:38:19.165870Z",
     "iopub.status.idle": "2026-10-09T08:38:19.228666Z",
     "shell.execute_reply": "2026-10-09T08:38:19.228666Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Largest mismatch (kg): 1.0\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th>Produced (good kg)</th>\n",
       "      <th>Sold kg</th>\n",
       "      <th>Expired kg</th>\n",
       "      <th>Closing stock</th>\n",
       "      <th>Opening stock</th>\n",
       "      <th>Check (should be 0)</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Roast</th>\n",
       "      <th>Month nr</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th rowspan=\"12\" valign=\"top\">Dark</th>\n",
       "      <th>1</th>\n",
       "      <td>30,720.00</td>\n",
       "      <td>27,802.00</td>\n",
       "      <td>0</td>\n",
       "      <td>18660</td>\n",
       "      <td>NaN</td>\n",
       "      <td>NaN</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>27,956.00</td>\n",
       "      <td>29,326.00</td>\n",
       "      <td>0</td>\n",
       "      <td>17290</td>\n",
       "      <td>18,660.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>32,650.00</td>\n",
       "      <td>22,710.00</td>\n",
       "      <td>0</td>\n",
       "      <td>27231</td>\n",
       "      <td>17,290.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>27,527.00</td>\n",
       "      <td>22,248.00</td>\n",
       "      <td>0</td>\n",
       "      <td>32511</td>\n",
       "      <td>27,231.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>27,009.00</td>\n",
       "      <td>22,838.00</td>\n",
       "      <td>0</td>\n",
       "      <td>36683</td>\n",
       "      <td>32,511.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>6</th>\n",
       "      <td>30,670.00</td>\n",
       "      <td>22,290.00</td>\n",
       "      <td>15000</td>\n",
       "      <td>30063</td>\n",
       "      <td>36,683.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>7</th>\n",
       "      <td>29,896.00</td>\n",
       "      <td>22,726.00</td>\n",
       "      <td>10000</td>\n",
       "      <td>27233</td>\n",
       "      <td>30,063.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>8</th>\n",
       "      <td>31,689.00</td>\n",
       "      <td>23,185.00</td>\n",
       "      <td>5000</td>\n",
       "      <td>30737</td>\n",
       "      <td>27,233.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>9</th>\n",
       "      <td>29,683.00</td>\n",
       "      <td>31,538.00</td>\n",
       "      <td>0</td>\n",
       "      <td>28883</td>\n",
       "      <td>30,737.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>10</th>\n",
       "      <td>28,618.00</td>\n",
       "      <td>32,898.00</td>\n",
       "      <td>0</td>\n",
       "      <td>24604</td>\n",
       "      <td>28,883.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>11</th>\n",
       "      <td>30,894.00</td>\n",
       "      <td>31,312.00</td>\n",
       "      <td>0</td>\n",
       "      <td>24186</td>\n",
       "      <td>24,604.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>12</th>\n",
       "      <td>27,427.00</td>\n",
       "      <td>32,490.00</td>\n",
       "      <td>0</td>\n",
       "      <td>19124</td>\n",
       "      <td>24,186.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th rowspan=\"12\" valign=\"top\">Light</th>\n",
       "      <th>1</th>\n",
       "      <td>13,784.00</td>\n",
       "      <td>12,050.00</td>\n",
       "      <td>0</td>\n",
       "      <td>5734</td>\n",
       "      <td>NaN</td>\n",
       "      <td>NaN</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>11,491.00</td>\n",
       "      <td>12,997.00</td>\n",
       "      <td>0</td>\n",
       "      <td>4228</td>\n",
       "      <td>5,734.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>13,651.00</td>\n",
       "      <td>10,304.00</td>\n",
       "      <td>0</td>\n",
       "      <td>7576</td>\n",
       "      <td>4,228.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>9,651.00</td>\n",
       "      <td>9,820.00</td>\n",
       "      <td>0</td>\n",
       "      <td>7407</td>\n",
       "      <td>7,576.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>13,055.00</td>\n",
       "      <td>10,234.00</td>\n",
       "      <td>0</td>\n",
       "      <td>10229</td>\n",
       "      <td>7,407.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>6</th>\n",
       "      <td>12,628.00</td>\n",
       "      <td>9,867.00</td>\n",
       "      <td>6000</td>\n",
       "      <td>6990</td>\n",
       "      <td>10,229.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>7</th>\n",
       "      <td>12,436.00</td>\n",
       "      <td>10,246.00</td>\n",
       "      <td>0</td>\n",
       "      <td>9181</td>\n",
       "      <td>6,990.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>8</th>\n",
       "      <td>13,649.00</td>\n",
       "      <td>10,090.00</td>\n",
       "      <td>0</td>\n",
       "      <td>12740</td>\n",
       "      <td>9,181.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>9</th>\n",
       "      <td>11,844.00</td>\n",
       "      <td>13,721.00</td>\n",
       "      <td>0</td>\n",
       "      <td>10863</td>\n",
       "      <td>12,740.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>10</th>\n",
       "      <td>13,375.00</td>\n",
       "      <td>14,018.00</td>\n",
       "      <td>0</td>\n",
       "      <td>10220</td>\n",
       "      <td>10,863.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>11</th>\n",
       "      <td>13,618.00</td>\n",
       "      <td>14,241.00</td>\n",
       "      <td>0</td>\n",
       "      <td>9597</td>\n",
       "      <td>10,220.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>12</th>\n",
       "      <td>10,568.00</td>\n",
       "      <td>14,117.00</td>\n",
       "      <td>0</td>\n",
       "      <td>6048</td>\n",
       "      <td>9,597.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th rowspan=\"12\" valign=\"top\">Medium</th>\n",
       "      <th>1</th>\n",
       "      <td>22,856.00</td>\n",
       "      <td>23,758.00</td>\n",
       "      <td>0</td>\n",
       "      <td>11349</td>\n",
       "      <td>NaN</td>\n",
       "      <td>NaN</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>21,533.00</td>\n",
       "      <td>23,852.00</td>\n",
       "      <td>0</td>\n",
       "      <td>9030</td>\n",
       "      <td>11,349.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>23,197.00</td>\n",
       "      <td>19,128.00</td>\n",
       "      <td>0</td>\n",
       "      <td>13100</td>\n",
       "      <td>9,030.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>21,290.00</td>\n",
       "      <td>18,754.00</td>\n",
       "      <td>0</td>\n",
       "      <td>15637</td>\n",
       "      <td>13,100.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>23,781.00</td>\n",
       "      <td>19,332.00</td>\n",
       "      <td>0</td>\n",
       "      <td>20087</td>\n",
       "      <td>15,637.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>6</th>\n",
       "      <td>22,979.00</td>\n",
       "      <td>18,785.00</td>\n",
       "      <td>0</td>\n",
       "      <td>24281</td>\n",
       "      <td>20,087.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>7</th>\n",
       "      <td>23,051.00</td>\n",
       "      <td>19,650.00</td>\n",
       "      <td>5000</td>\n",
       "      <td>22682</td>\n",
       "      <td>24,281.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>8</th>\n",
       "      <td>25,357.00</td>\n",
       "      <td>18,957.00</td>\n",
       "      <td>7000</td>\n",
       "      <td>22083</td>\n",
       "      <td>22,682.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>9</th>\n",
       "      <td>24,161.00</td>\n",
       "      <td>24,994.00</td>\n",
       "      <td>0</td>\n",
       "      <td>21251</td>\n",
       "      <td>22,083.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>10</th>\n",
       "      <td>22,472.00</td>\n",
       "      <td>25,796.00</td>\n",
       "      <td>0</td>\n",
       "      <td>17928</td>\n",
       "      <td>21,251.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>11</th>\n",
       "      <td>23,406.00</td>\n",
       "      <td>25,088.00</td>\n",
       "      <td>0</td>\n",
       "      <td>16246</td>\n",
       "      <td>17,928.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>12</th>\n",
       "      <td>20,145.00</td>\n",
       "      <td>26,104.00</td>\n",
       "      <td>0</td>\n",
       "      <td>10288</td>\n",
       "      <td>16,246.00</td>\n",
       "      <td>1.00</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                 Produced (good kg)   Sold kg  Expired kg  Closing stock  \\\n",
       "Roast  Month nr                                                            \n",
       "Dark   1                  30,720.00 27,802.00           0          18660   \n",
       "       2                  27,956.00 29,326.00           0          17290   \n",
       "       3                  32,650.00 22,710.00           0          27231   \n",
       "       4                  27,527.00 22,248.00           0          32511   \n",
       "       5                  27,009.00 22,838.00           0          36683   \n",
       "       6                  30,670.00 22,290.00       15000          30063   \n",
       "       7                  29,896.00 22,726.00       10000          27233   \n",
       "       8                  31,689.00 23,185.00        5000          30737   \n",
       "       9                  29,683.00 31,538.00           0          28883   \n",
       "       10                 28,618.00 32,898.00           0          24604   \n",
       "       11                 30,894.00 31,312.00           0          24186   \n",
       "       12                 27,427.00 32,490.00           0          19124   \n",
       "Light  1                  13,784.00 12,050.00           0           5734   \n",
       "       2                  11,491.00 12,997.00           0           4228   \n",
       "       3                  13,651.00 10,304.00           0           7576   \n",
       "       4                   9,651.00  9,820.00           0           7407   \n",
       "       5                  13,055.00 10,234.00           0          10229   \n",
       "       6                  12,628.00  9,867.00        6000           6990   \n",
       "       7                  12,436.00 10,246.00           0           9181   \n",
       "       8                  13,649.00 10,090.00           0          12740   \n",
       "       9                  11,844.00 13,721.00           0          10863   \n",
       "       10                 13,375.00 14,018.00           0          10220   \n",
       "       11                 13,618.00 14,241.00           0           9597   \n",
       "       12                 10,568.00 14,117.00           0           6048   \n",
       "Medium 1                  22,856.00 23,758.00           0          11349   \n",
       "       2                  21,533.00 23,852.00           0           9030   \n",
       "       3                  23,197.00 19,128.00           0          13100   \n",
       "       4                  21,290.00 18,754.00           0          15637   \n",
       "       5                  23,781.00 19,332.00           0          20087   \n",
       "       6                  22,979.00 18,785.00           0          24281   \n",
       "       7                  23,051.00 19,650.00        5000          22682   \n",
       "       8                  25,357.00 18,957.00        7000          22083   \n",
       "       9                  24,161.00 24,994.00           0          21251   \n",
       "       10                 22,472.00 25,796.00           0          17928   \n",
       "       11                 23,406.00 25,088.00           0          16246   \n",
       "       12                 20,145.00 26,104.00           0          10288   \n",
       "\n",
       "                 Opening stock  Check (should be 0)  \n",
       "Roast  Month nr                                      \n",
       "Dark   1                   NaN                  NaN  \n",
       "       2             18,660.00                 0.00  \n",
       "       3             17,290.00                 1.00  \n",
       "       4             27,231.00                 0.00  \n",
       "       5             32,511.00                 1.00  \n",
       "       6             36,683.00                 0.00  \n",
       "       7             30,063.00                 1.00  \n",
       "       8             27,233.00                 0.00  \n",
       "       9             30,737.00                 1.00  \n",
       "       10            28,883.00                 0.00  \n",
       "       11            24,604.00                 1.00  \n",
       "       12            24,186.00                 1.00  \n",
       "Light  1                   NaN                  NaN  \n",
       "       2              5,734.00                 0.00  \n",
       "       3              4,228.00                 1.00  \n",
       "       4              7,576.00                 0.00  \n",
       "       5              7,407.00                 1.00  \n",
       "       6             10,229.00                 0.00  \n",
       "       7              6,990.00                 1.00  \n",
       "       8              9,181.00                 0.00  \n",
       "       9             12,740.00                 0.00  \n",
       "       10            10,863.00                 0.00  \n",
       "       11            10,220.00                 0.00  \n",
       "       12             9,597.00                 0.00  \n",
       "Medium 1                   NaN                  NaN  \n",
       "       2             11,349.00                 1.00  \n",
       "       3              9,030.00                 1.00  \n",
       "       4             13,100.00                 1.00  \n",
       "       5             15,637.00                 1.00  \n",
       "       6             20,087.00                 0.00  \n",
       "       7             24,281.00                 0.00  \n",
       "       8             22,682.00                 1.00  \n",
       "       9             22,083.00                 1.00  \n",
       "       10            21,251.00                 1.00  \n",
       "       11            17,928.00                 0.00  \n",
       "       12            16,246.00                 1.00  "
      ]
     },
     "execution_count": 7,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "sales_kg = (pd.concat([ec[[\"Roast\", \"Month nr\", \"Kg\"]], b2b[[\"Roast\", \"Month nr\", \"Kg\"]]])\n",
    "              .groupby([\"Roast\", \"Month nr\"])[\"Kg\"].sum())\n",
    "flow = pd.DataFrame({\n",
    "    \"Produced (good kg)\": ro.groupby([\"Roast\", \"Month nr\"])[\"Kg good\"].sum(),\n",
    "    \"Sold kg\": sales_kg,\n",
    "    \"Expired kg\": fin.set_index([\"Roast\", \"Month nr\"])[\"Scrapped\"],\n",
    "    \"Closing stock\": fin.set_index([\"Roast\", \"Month nr\"])[\"Stock level\"],\n",
    "})\n",
    "flow[\"Opening stock\"] = flow.groupby(level=\"Roast\")[\"Closing stock\"].shift()\n",
    "flow[\"Check (should be 0)\"] = flow[\"Closing stock\"] - (\n",
    "    flow[\"Opening stock\"] + flow[\"Produced (good kg)\"] - flow[\"Sold kg\"] - flow[\"Expired kg\"])\n",
    "print(\"Largest mismatch (kg):\", flow[\"Check (should be 0)\"].abs().max().round(1))\n",
    "flow.round(0)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "001a8af2",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-09T08:38:19.230073Z",
     "iopub.status.busy": "2026-10-09T08:38:19.230073Z",
     "iopub.status.idle": "2026-10-09T08:38:19.403346Z",
     "shell.execute_reply": "2026-10-09T08:38:19.403346Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Coffee bag w/valve 0,5</th>\n",
       "      <th>Coffee bag w/valve 1,0</th>\n",
       "      <th>Shipping box</th>\n",
       "      <th>0.5 kg bags sold</th>\n",
       "      <th>1 kg bags sold</th>\n",
       "      <th>E-com orders (boxes)</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Month nr</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>20,000.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>20,000.00</td>\n",
       "      <td>90435</td>\n",
       "      <td>20958</td>\n",
       "      <td>18438</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>20,000.00</td>\n",
       "      <td>10,000.00</td>\n",
       "      <td>20,000.00</td>\n",
       "      <td>70806</td>\n",
       "      <td>16738</td>\n",
       "      <td>20413</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>10,000.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>20,000.00</td>\n",
       "      <td>69344</td>\n",
       "      <td>16150</td>\n",
       "      <td>19755</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>20,000.00</td>\n",
       "      <td>10,000.00</td>\n",
       "      <td>20,000.00</td>\n",
       "      <td>71043</td>\n",
       "      <td>16882</td>\n",
       "      <td>20414</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>6</th>\n",
       "      <td>10,000.00</td>\n",
       "      <td>10,000.00</td>\n",
       "      <td>20,000.00</td>\n",
       "      <td>69937</td>\n",
       "      <td>15974</td>\n",
       "      <td>19755</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>7</th>\n",
       "      <td>10,000.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>20,000.00</td>\n",
       "      <td>71660</td>\n",
       "      <td>16793</td>\n",
       "      <td>20413</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>8</th>\n",
       "      <td>20,000.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>20,000.00</td>\n",
       "      <td>71656</td>\n",
       "      <td>16404</td>\n",
       "      <td>20414</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>9</th>\n",
       "      <td>20,000.00</td>\n",
       "      <td>10,000.00</td>\n",
       "      <td>20,000.00</td>\n",
       "      <td>96754</td>\n",
       "      <td>21876</td>\n",
       "      <td>19755</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>10</th>\n",
       "      <td>10,000.00</td>\n",
       "      <td>10,000.00</td>\n",
       "      <td>20,000.00</td>\n",
       "      <td>99757</td>\n",
       "      <td>22833</td>\n",
       "      <td>20413</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>11</th>\n",
       "      <td>20,000.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>20,000.00</td>\n",
       "      <td>97756</td>\n",
       "      <td>21764</td>\n",
       "      <td>19755</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>12</th>\n",
       "      <td>10,000.00</td>\n",
       "      <td>10,000.00</td>\n",
       "      <td>20,000.00</td>\n",
       "      <td>99709</td>\n",
       "      <td>22856</td>\n",
       "      <td>20414</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "          Coffee bag w/valve 0,5  Coffee bag w/valve 1,0  Shipping box  \\\n",
       "Month nr                                                                 \n",
       "2                      20,000.00                    0.00     20,000.00   \n",
       "3                      20,000.00               10,000.00     20,000.00   \n",
       "4                      10,000.00                    0.00     20,000.00   \n",
       "5                      20,000.00               10,000.00     20,000.00   \n",
       "6                      10,000.00               10,000.00     20,000.00   \n",
       "7                      10,000.00                    0.00     20,000.00   \n",
       "8                      20,000.00                    0.00     20,000.00   \n",
       "9                      20,000.00               10,000.00     20,000.00   \n",
       "10                     10,000.00               10,000.00     20,000.00   \n",
       "11                     20,000.00                    0.00     20,000.00   \n",
       "12                     10,000.00               10,000.00     20,000.00   \n",
       "\n",
       "          0.5 kg bags sold  1 kg bags sold  E-com orders (boxes)  \n",
       "Month nr                                                          \n",
       "2                    90435           20958                 18438  \n",
       "3                    70806           16738                 20413  \n",
       "4                    69344           16150                 19755  \n",
       "5                    71043           16882                 20414  \n",
       "6                    69937           15974                 19755  \n",
       "7                    71660           16793                 20413  \n",
       "8                    71656           16404                 20414  \n",
       "9                    96754           21876                 19755  \n",
       "10                   99757           22833                 20413  \n",
       "11                   97756           21764                 19755  \n",
       "12                   99709           22856                 20414  "
      ]
     },
     "execution_count": 8,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "pack_used = raw[~beans].pivot(index=\"Month nr\", columns=\"Product\", values=\"Used\")\n",
    "shipped = pd.DataFrame({\n",
    "    \"0.5 kg bags sold\": pd.concat([ec, b2b]).query(\"`Bag kg` == 0.5\").groupby(\"Month nr\")[\"Quantity\"].sum(),\n",
    "    \"1 kg bags sold\": pd.concat([ec, b2b]).query(\"`Bag kg` == 1.0\").groupby(\"Month nr\")[\"Quantity\"].sum(),\n",
    "    \"E-com orders (boxes)\": ec.groupby(\"Month nr\").size(),\n",
    "})\n",
    "pack_used.join(shipped).loc[2:]"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "da89d810",
   "metadata": {},
   "source": [
    "**Result:**\n",
    "- Green beans and finished product reconcile **exactly** (differences ≤ 1 kg from rounding). The data is internally consistent, and production = (batches − rejected batches) × final weight.\n",
    "- Shipping boxes match: **1 box per e-com order** (≈ 20,000 a month).\n",
    "- **Coffee bags do not match.** The inventory shows 10,000–20,000 bags used per month, but ≈ 70,000–100,000 half-kilo and ≈ 16,000–23,000 one-kilo bags are sold. Either the packaging records are incomplete, or bag stock is a hidden risk. **Ask the teacher in the Week 6 Q&A.**"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "46cfa277",
   "metadata": {},
   "source": [
    "## 6. First look per operations step"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "id": "2bbe1cff",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-09T08:38:19.403346Z",
     "iopub.status.busy": "2026-10-09T08:38:19.403346Z",
     "iopub.status.idle": "2026-10-09T08:38:19.446775Z",
     "shell.execute_reply": "2026-10-09T08:38:19.446775Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "=== SOURCING: green-bean containers per origin (1 container = 10,080 kg) ===\n",
      "            containers  months_with_delivery  price_per_kg\n",
      "Origin                                                    \n",
      "Colombia         46.00                    12          6.95\n",
      "Costa Rica       18.00                     9         11.50\n",
      "Uganda           34.00                    12          8.50\n",
      "\n",
      "=== RAW-MATERIAL STORAGE: scrapped ===\n",
      "                    Product Month  Scrapped Reasoncode  Pallets scrapped  Scrapped value\n",
      "Coffee Beans (COO Colombia)   May      1680        EXP              2.00       11,676.00\n",
      "  Coffee Beans (COO Uganda)   May      3360        EXP              4.00       28,560.00\n",
      "     Coffee bag w/valve 0,5  June     10000        DAM               NaN        4,000.00\n",
      "               Shipping box March      5000        DAM               NaN        4,550.00\n",
      "\n",
      "=== ROASTING: throughput vs rated capacity ===\n",
      "                hours batch_fill  hours per workday  actual kg/h  rated kg/h vs rated\n",
      "Machine                                                                              \n",
      "Probat P60   2,947.70        91%              11.60       113.20         215      53%\n",
      "Probat Px120 3,193.00        92%              12.60       198.20         480      41%\n",
      "        Batches  Rejected batches  QC weight  QC color  QC profile Reject rate  Yield\n",
      "Roast                                                                                \n",
      "Dark   4,076.00                63         25        18          20        1.5%  80.2%\n",
      "Light  1,672.00                83         27        27          29        5.0%  85.9%\n",
      "Medium 6,090.00                78         20        34          24        1.3%  83.2%\n",
      "Green beans in: 967 t -> roasted out: 796 t (profile says ±1,000 t)\n",
      "\n",
      "=== COOLING: share of each batch cycle the roaster is not roasting ===\n",
      "        Setup min  Run min  Cooling min Not roasting\n",
      "Roast                                               \n",
      "Dark         6.30    17.10        11.10          50%\n",
      "Light        5.70    15.00         9.90          51%\n",
      "Medium       4.10    16.00         9.00          45%\n",
      "\n",
      "=== PACKING ===\n",
      "Packing line: 8.3 hours per workday\n",
      "\n",
      "=== FINISHED-PRODUCT STORAGE: expired stock ===\n",
      " Roast  Month  Scrapped  Scrapped value\n",
      "  Dark   June     15000      344,700.00\n",
      "  Dark   July     10000      229,800.00\n",
      "  Dark August      5000      114,900.00\n",
      "Medium   July      5000      129,500.00\n",
      "Medium August      7000      181,300.00\n",
      " Light   June      6000      159,000.00\n",
      "Total: 48,000 kg = EUR 1,159,200 at sales price\n",
      "\n",
      "=== DELIVERY ===\n",
      "E-com revenue EUR 17,561,220 | B2B revenue EUR 421,652 (21 customers, roasts: ['Medium'])\n",
      "On time (next business day): 91.1%\n",
      "Orders > EUR 55 still charged a fee: 30,725 (EUR 152,089)\n",
      "Net shipping cost paid by company: EUR 1,023,233\n"
     ]
    }
   ],
   "source": [
    "def h(t): print(f\"\\n=== {t} ===\")\n",
    "\n",
    "h(\"SOURCING: green-bean containers per origin (1 container = 10,080 kg)\")\n",
    "print(raw[beans].groupby(\"Origin\").agg(\n",
    "    containers=(\"Containers purchased\", \"sum\"),\n",
    "    months_with_delivery=(\"Purchased\", lambda s: int((s > 0).sum())),\n",
    "    price_per_kg=(\"Unit price\", \"first\")).to_string())\n",
    "\n",
    "h(\"RAW-MATERIAL STORAGE: scrapped\")\n",
    "print(raw[raw[\"Scrapped\"] > 0][[\"Product\", \"Month\", \"Scrapped\", \"Reasoncode\",\n",
    "                                \"Pallets scrapped\", \"Scrapped value\"]].to_string(index=False))\n",
    "\n",
    "h(\"ROASTING: throughput vs rated capacity\")\n",
    "m = ro.groupby(\"Machine\").agg(hours=(\"Roaster hours\", \"sum\"), kg=(\"Kg loaded\", \"sum\"),\n",
    "                              batch_fill=(\"Batch fill\", \"mean\"))\n",
    "m[\"hours per workday\"] = m[\"hours\"] / ro.groupby([\"Machine\", \"Month\"])[\"Workdays\"].first().groupby(\"Machine\").sum()\n",
    "m[\"actual kg/h\"] = m[\"kg\"] / m[\"hours\"]\n",
    "m[\"rated kg/h\"] = ROASTER_SPECS[\"Rated kg/h\"]\n",
    "m[\"vs rated\"] = (m[\"actual kg/h\"] / m[\"rated kg/h\"]).map(\"{:.0%}\".format)\n",
    "m[\"batch_fill\"] = m[\"batch_fill\"].map(\"{:.0%}\".format)\n",
    "print(m.drop(columns=\"kg\").round(1).to_string())\n",
    "q = ro.groupby(\"Roast\")[[\"Batches\", \"Rejected batches\", \"QC weight\", \"QC color\", \"QC profile\"]].sum()\n",
    "q[\"Reject rate\"] = q[\"Rejected batches\"] / q[\"Batches\"]\n",
    "q[\"Yield\"] = ro.groupby(\"Roast\")[\"Kg roasted\"].sum() / ro.groupby(\"Roast\")[\"Kg loaded\"].sum()\n",
    "q[[\"Reject rate\", \"Yield\"]] = q[[\"Reject rate\", \"Yield\"]].map(\"{:.1%}\".format)\n",
    "print(q.to_string())\n",
    "print(f\"Green beans in: {ro['Kg loaded'].sum()/1000:,.0f} t -> roasted out: \"\n",
    "      f\"{ro['Kg roasted'].sum()/1000:,.0f} t (profile says ±{STATED_OUTPUT_T:,} t)\")\n",
    "\n",
    "h(\"COOLING: share of each batch cycle the roaster is not roasting\")\n",
    "c = ro.groupby(\"Roast\")[[\"Setup min\", \"Run min\", \"Cooling min\"]].mean()\n",
    "c[\"Not roasting\"] = (1 - c[\"Run min\"] / c.sum(axis=1)).map(\"{:.0%}\".format)\n",
    "print(c.round(1).to_string())\n",
    "\n",
    "h(\"PACKING\")\n",
    "print(f\"Packing line: {ro['Packing hours'].sum() / 254:.1f} hours per workday\")\n",
    "\n",
    "h(\"FINISHED-PRODUCT STORAGE: expired stock\")\n",
    "print(fin[fin[\"Scrapped\"] > 0][[\"Roast\", \"Month\", \"Scrapped\", \"Scrapped value\"]].to_string(index=False))\n",
    "print(f\"Total: {fin['Scrapped'].sum():,.0f} kg = EUR {fin['Scrapped value'].sum():,.0f} at sales price\")\n",
    "\n",
    "h(\"DELIVERY\")\n",
    "print(f\"E-com revenue EUR {ec['Value'].sum():,.0f} | B2B revenue EUR {b2b['Value'].sum():,.0f} \"\n",
    "      f\"({b2b['Customer'].nunique()} customers, roasts: {b2b['Roast'].unique().tolist()})\")\n",
    "print(f\"On time (next business day): {ec['On time'].mean():.1%}\")\n",
    "print(f\"Orders > EUR {FREE_SHIPPING_ABOVE} still charged a fee: {ec['Wrongly charged'].sum():,} \"\n",
    "      f\"(EUR {ec.loc[ec['Wrongly charged'], 'Delivery fee'].sum():,.0f})\")\n",
    "print(f\"Net shipping cost paid by company: EUR {ec['Net shipping cost'].sum():,.0f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e6fd77f4",
   "metadata": {},
   "source": [
    "### Steady state & seasonality: orders are flat, but order size is not"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "8499f0d7",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-09T08:38:19.449478Z",
     "iopub.status.busy": "2026-10-09T08:38:19.449478Z",
     "iopub.status.idle": "2026-10-09T08:38:19.477106Z",
     "shell.execute_reply": "2026-10-09T08:38:19.476698Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Orders per day</th>\n",
       "      <th>Kg sold</th>\n",
       "      <th>Kg produced</th>\n",
       "      <th>Finished stock (end)</th>\n",
       "      <th>Produced - sold</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Month nr</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>658.00</td>\n",
       "      <td>63,611.00</td>\n",
       "      <td>67,361.00</td>\n",
       "      <td>35743</td>\n",
       "      <td>3,750.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>658.00</td>\n",
       "      <td>66,176.00</td>\n",
       "      <td>60,979.00</td>\n",
       "      <td>30548</td>\n",
       "      <td>-5,196.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>658.00</td>\n",
       "      <td>52,141.00</td>\n",
       "      <td>69,498.00</td>\n",
       "      <td>47907</td>\n",
       "      <td>17,357.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>658.00</td>\n",
       "      <td>50,822.00</td>\n",
       "      <td>58,468.00</td>\n",
       "      <td>55555</td>\n",
       "      <td>7,646.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>659.00</td>\n",
       "      <td>52,404.00</td>\n",
       "      <td>63,845.00</td>\n",
       "      <td>66999</td>\n",
       "      <td>11,441.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>6</th>\n",
       "      <td>658.00</td>\n",
       "      <td>50,942.00</td>\n",
       "      <td>66,277.00</td>\n",
       "      <td>61334</td>\n",
       "      <td>15,334.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>7</th>\n",
       "      <td>658.00</td>\n",
       "      <td>52,623.00</td>\n",
       "      <td>65,384.00</td>\n",
       "      <td>59096</td>\n",
       "      <td>12,761.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>8</th>\n",
       "      <td>659.00</td>\n",
       "      <td>52,232.00</td>\n",
       "      <td>70,695.00</td>\n",
       "      <td>65560</td>\n",
       "      <td>18,463.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>9</th>\n",
       "      <td>658.00</td>\n",
       "      <td>70,253.00</td>\n",
       "      <td>65,688.00</td>\n",
       "      <td>60997</td>\n",
       "      <td>-4,565.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>10</th>\n",
       "      <td>658.00</td>\n",
       "      <td>72,712.00</td>\n",
       "      <td>64,465.00</td>\n",
       "      <td>52752</td>\n",
       "      <td>-8,246.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>11</th>\n",
       "      <td>658.00</td>\n",
       "      <td>70,642.00</td>\n",
       "      <td>67,918.00</td>\n",
       "      <td>50029</td>\n",
       "      <td>-2,724.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>12</th>\n",
       "      <td>659.00</td>\n",
       "      <td>72,710.00</td>\n",
       "      <td>58,140.00</td>\n",
       "      <td>35460</td>\n",
       "      <td>-14,571.00</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "          Orders per day   Kg sold  Kg produced  Finished stock (end)  \\\n",
       "Month nr                                                                \n",
       "1                 658.00 63,611.00    67,361.00                 35743   \n",
       "2                 658.00 66,176.00    60,979.00                 30548   \n",
       "3                 658.00 52,141.00    69,498.00                 47907   \n",
       "4                 658.00 50,822.00    58,468.00                 55555   \n",
       "5                 659.00 52,404.00    63,845.00                 66999   \n",
       "6                 658.00 50,942.00    66,277.00                 61334   \n",
       "7                 658.00 52,623.00    65,384.00                 59096   \n",
       "8                 659.00 52,232.00    70,695.00                 65560   \n",
       "9                 658.00 70,253.00    65,688.00                 60997   \n",
       "10                658.00 72,712.00    64,465.00                 52752   \n",
       "11                658.00 70,642.00    67,918.00                 50029   \n",
       "12                659.00 72,710.00    58,140.00                 35460   \n",
       "\n",
       "          Produced - sold  \n",
       "Month nr                   \n",
       "1                3,750.00  \n",
       "2               -5,196.00  \n",
       "3               17,357.00  \n",
       "4                7,646.00  \n",
       "5               11,441.00  \n",
       "6               15,334.00  \n",
       "7               12,761.00  \n",
       "8               18,463.00  \n",
       "9               -4,565.00  \n",
       "10              -8,246.00  \n",
       "11              -2,724.00  \n",
       "12             -14,571.00  "
      ]
     },
     "execution_count": 10,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "monthly = pd.DataFrame({\n",
    "    \"Orders per day\": ec.groupby(\"Month nr\").size() / ec.groupby(\"Month nr\")[\"Order date\"].nunique(),\n",
    "    \"Kg sold\": sales_kg.groupby(level=\"Month nr\").sum(),\n",
    "    \"Kg produced\": ro.groupby(\"Month nr\")[\"Kg good\"].sum(),\n",
    "    \"Finished stock (end)\": fin.groupby(\"Month nr\")[\"Stock level\"].sum(),\n",
    "})\n",
    "monthly[\"Produced - sold\"] = monthly[\"Kg produced\"] - monthly[\"Kg sold\"]\n",
    "monthly.round(0)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4a6e2d53",
   "metadata": {},
   "source": [
    "## 7. Save the clean data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "id": "ae4d7a21",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-09T08:38:19.480458Z",
     "iopub.status.busy": "2026-10-09T08:38:19.480458Z",
     "iopub.status.idle": "2026-10-09T08:38:21.179019Z",
     "shell.execute_reply": "2026-10-09T08:38:21.179019Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Saved to C:\\Users\\floyd\\AppData\\Roaming\\Claude\\scratch-workspaces\\3cf67011-d217-456e-927e-9c7020195e99\\edb7be3c-9c78-471c-9f96-3e7c16929c9d\\scratch-2026-10-09-3f62ce\\bean-there-done-that\\data\\clean\n"
     ]
    }
   ],
   "source": [
    "raw.to_csv(CLEAN / \"raw_material.csv\", index=False)\n",
    "ro.to_csv(CLEAN / \"roasting.csv\", index=False)\n",
    "fin.to_csv(CLEAN / \"finished_product.csv\", index=False)\n",
    "ec.to_csv(CLEAN / \"ecom_orders.csv\", index=False)\n",
    "b2b.to_csv(CLEAN / \"b2b_orders.csv\", index=False)\n",
    "coverage.to_csv(CLEAN / \"operations_coverage.csv\", index=False)\n",
    "print(\"Saved to\", CLEAN)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "96328e50",
   "metadata": {},
   "source": [
    "## 8. Summary of Step 1\n",
    "\n",
    "**Data quality:** the sheets are consistent. Bean and finished-goods stock flows reconcile to the kilo. The exception is coffee-bag packaging (see section 5).\n",
    "\n",
    "**What we learned per operations step (to be turned into KPIs in Step 2):**\n",
    "\n",
    "| Step | First finding |\n",
    "|---|---|\n",
    "| Sourcing | Beans arrive in whole containers (98 in 2023). One farm per origin, no backup. Costa Rica skipped 3 months. |\n",
    "| Raw-material storage | 6 pallets of green beans **expired** in May (≈ €40k), so beans are bought too early or too much. 10,000 bags and 5,000 boxes damaged. Bag stock does not match bag usage. |\n",
    "| Roasting | Machines reach only **41–53% of rated kg/h**, even though batches are ~92% full. Light roast rejects 5% of batches (others ~1.4%). |\n",
    "| Cooling | Setup + cooling = **~half of every batch cycle**. If cooling runs while the next batch roasts, there is a lot of hidden capacity. |\n",
    "| Packing | ~8.3 h per workday: fine today, a possible limit when growing. |\n",
    "| Finished-product storage | 48 t expired (≈ €1.16M). **Root cause:** demand drops ~25% from March to August (≈ 52 t vs ≈ 70 t a month), but production stays flat at ≈ 65 t, so ≈ 83 t extra stock builds up and part of it expires, so stock builds up and expires. |\n",
    "| Delivery | 91% on time (business days). €1.0M net shipping subsidy. 30,725 orders over €55 wrongly charged a fee. B2B is small (21 clients, Medium roast only). |\n",
    "\n",
    "**Questions for the teacher (Week 6 Q&A):**\n",
    "1. Is the roaster idle during cooling, or can the next batch start while the previous one cools?\n",
    "2. Is the coffee-bag inventory complete? It cannot cover the bags sold.\n",
    "3. Does \"±1,000 t\" mean green beans in (967 t) or roasted coffee out (796 t)?\n",
    "\n",
    "**Next: Step 2**, KPIs per operations step and the growth simulation."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.12.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}


{'cells': [{'cell_type': 'markdown',
   'id': '435bf210',
   'metadata': {},
   'source': ['# Step 1 — Business & data understanding, data preparation\n',
    '**Bean there done that · Group 7** · CRISP-DM phases 1–3\n',
    '\n',
    'This notebook:\n',
    '1. writes down what we know about the business (from the Week 3 assignment slides),\n',
    "2. maps every sheet of the dataset to a step in the company's operations,\n",
    '3. cleans the data and saves it to `data/clean/` (all later steps start from there),\n',
    '4. checks whether the data is consistent, and lists what we found and what we still need to ask.\n',
    '\n',
    '**Before running:** upload `Bean there done that data.xlsx` into this folder (ideally `data/raw/`).']},
  {'cell_type': 'markdown',
   'id': 'ef9fde46',
   'metadata': {},
   'source': ['## 1. Business understanding\n',
    '\n',
    '**The question:** *The CEO wants to know if Bean there done that is ready to grow as of next year.*\n',
    'The compan